<a href="https://colab.research.google.com/github/SudhaDevi17/LargeLanguageModels/blob/main/jev_decision_primitives_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

-
# Jev Decision Primitives & OpenRouter API Integration

This notebook demonstrates how to evaluate structured decision-making primitives (**Choice**, **Score**, and **Noul**) using TypeSafe's Jev model engine (`typesafe/jev-1.13`) hosted via the OpenRouter Alpha API.

### Key Primitives Covered
- **Choice:** Single-class categorical classification with probability distributions.
- **Score:** Ordered discrete/continuous rating across defined levels.
- **Noul:** Calibrated binary probability ($P(\text{Yes})$).

- Get an API key from the [TypeSafe dashboard](https://console.typesafe.ai)
---

In [39]:
# Install SDK dependency quietly
!pip install -q typesafe-sdk

import os
import requests
from google.colab import userdata

# Retrieve API Key securely from Colab Secrets
OPENROUTER_API_KEY = userdata.get("OPEN_ROUTER_API_KEY")
os.environ["OPEN_ROUTER_API_KEY"] = OPENROUTER_API_KEY

API_URL = "https://openrouter.ai/api/alpha/decisions"
HEADERS = {
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
    "Content-Type": "application/json",
}

## 1. Categorical Classification (`Choice`)

Use `Choice` when an input must be mapped to exactly one category out of a predefined list.

In [40]:
# Setup ticket payload
ticket_state = {
    "customer_tier": "enterprise",
    "ticket": "The Stripe integration has failed for three days. I am losing sales. Please help ASAP.",
}

payload_choice = {
    "model": "typesafe/jev-1.13",
    "state": ticket_state,
    "questions": {
        "department": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {
                "billing": "Payment or subscription issues",
                "technical": "Bugs or integration problems",
                "sales": "Pricing or account questions",
            },
        }
    },
}

response = requests.post(API_URL, headers=HEADERS, json=payload_choice)
answers = response.json().get("answers", {})

# Clean output presentation
print("Selected Department:", answers["department"]["choice"])
print("Confidence:", answers["department"]["confidence"])
print("Probabilities:", answers["department"]["probabilities"])

Selected Department: technical
Confidence: 0.85
Probabilities: {'sales': 0, 'billing': 0.1, 'technical': 0.9}


## 2. Severity Evaluation (`Score`)

Use `Score` when evaluating inputs across ordered, descriptive thresholds (e.g., Low $\rightarrow$ Medium $\rightarrow$ High).

In [41]:
bug_state = {
    "customer_tier": "enterprise",
    "ticket": "The export button crashes the settings page in Safari. It works in Chrome, but some customers only use Safari.",
}

payload_score = {
    "model": "typesafe/jev-1.13",
    "state": bug_state,
    "questions": {
        "severity": {
            "type": "score",
            "instructions": "How severe is the reported issue?",
            "criteria": [
                "Cosmetic; no functional impact",
                "Broken feature, but a workaround exists",
                "Blocking issue; no workaround exists",
            ],
        }
    },
}

response = requests.post(API_URL, headers=HEADERS, json=payload_score)
answers = response.json().get("answers", {})

print("Estimated Score:", answers["severity"]["score"])
print("Confidence:", answers["severity"]["confidence"])
print("Distribution:", answers["severity"]["probabilities"])

Estimated Score: 1.25
Confidence: 0.63
Distribution: {'0': 0, '1': 0.75, '2': 0.25}


## 3. Binary Probability (`Noul`)

Use `Noul` for binary true/false queries. It returns a calibrated scalar probability $P(\text{True})$.

In [42]:
escalation_state = {
    "customer_tier": "enterprise",
    "ticket": "I have asked three times now. Can I please just talk to a real person?",
}

payload_noul = {
    "model": "typesafe/jev-1.13",
    "state": escalation_state,
    "questions": {
        "is_human_escalation": {
            "type": "noul",
            "instructions": "Is the customer asking for a human agent?",
        }
    },
}

response = requests.post(API_URL, headers=HEADERS, json=payload_noul)
answers = response.json().get("answers", {})

prob_yes = answers["is_human_escalation"]["noul"]
print(f"Human Escalation Probability: {prob_yes:.2%}")

Human Escalation Probability: 99.00%


## 4. Multi-Primitive Evaluation Pipeline

Evaluate multiple decision primitives in a single API request against shared context.

In [43]:
multi_payload = {
    "model": "typesafe/jev-1.13",
    "state": {
        "customer_tier": "enterprise",
        "ticket": "I have asked three times now. Can I please just talk to a real person?",
    },
    "questions": {
        "is_human_escalation": {
            "type": "noul",
            "instructions": "Is the customer asking for a human agent?",
        },
        "department": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {
                "billing": "Payment or subscription issues",
                "technical": "Bugs or integration problems",
                "sales": "Pricing or account questions",
                "general_support": "Vague complaints, general inquiry, or simple human requests",
            },
        },
        "severity": {
            "type": "score",
            "instructions": "How severe is the reported issue?",
            "criteria": [
                "Cosmetic; no functional impact",
                "Broken feature, but a workaround exists",
                "Blocking issue; no workaround exists",
            ],
        },
        "is_urgent": {
            "type": "noul",
            "instructions": "The message conveys urgency or time-sensitivity",
        },
    },
}

response = requests.post(API_URL, headers=HEADERS, json=multi_payload)
res_data = response.json()

print(f"Model ID: {res_data.get('model')}")
print(f"Total Cost: ${res_data.get('usage', {}).get('cost', 0):.6f}\n")
print("Evaluation Results:")
for question_key, val in res_data.get("answers", {}).items():
  print(f" - {question_key}: {val}")

Model ID: typesafe/jev-1.13-20260917
Total Cost: $0.000020

Evaluation Results:
 - is_human_escalation: {'type': 'noul', 'noul': 0.99}
 - department: {'type': 'choice', 'choice': 'general_support', 'probabilities': {'sales': 0, 'technical': 0, 'billing': 0, 'general_support': 1}, 'confidence': 1}
 - severity: {'type': 'score', 'score': 1.07, 'legend': {'0': 'Cosmetic; no functional impact', '1': 'Broken feature, but a workaround exists', '2': 'Blocking issue; no workaround exists'}, 'probabilities': {'0': 0.44, '1': 0.06, '2': 0.5}, 'confidence': 0}
 - is_urgent: {'type': 'noul', 'noul': 0.77}


Read the [TypeSafe Quick start](https://docs.typesafe.ai/introduction/quickstart), [Choice](https://docs.typesafe.ai/primitives/choice), [Score](https://docs.typesafe.ai/primitives/score), and [Noul](https://docs.typesafe.ai/primitives/noul) documentation for API details.